In [2]:
import pandas as pd
import re
from pathlib import Path

## Cleaning of Jiji.ng Dataset

In [3]:
BASE_DIR = Path.cwd().parent  
RAW = BASE_DIR / "Data" / "raw"
df_original = pd.read_csv(RAW / "jiji_abuja_rentals.csv")

In [4]:
df_original.head()

,Bedrooms,Bedrooms(category),Property Category,Price,Price(with currency),District,Scraped_At
0,1,1 bedroom,"Studio Apartment in Arab Road, Kubwa for rent",600000,"₦ 600,000",Kubwa,2026-05-30 13:25:47
1,1,1 bedroom,"1bdrm Block of Flats in Arab Road, Kubwa for rent",1500000,"₦ 1,500,000",Kubwa,2026-05-30 13:25:47
2,1,1 bedroom,"Studio Apartment in Efab City Estate, Life Cam...",1200000,"₦ 1,200,000",Life Camp,2026-05-30 13:25:47
3,1,1 bedroom,Furnished 1bdrm Apartment in Las Colinas Villa...,1250000,"₦ 1,250,000",Bunkoro,2026-05-30 13:25:47
4,1,1 bedroom,"1bdrm Apartment in Arab Road, Kubwa for rent",1200000,"₦ 1,200,000",Kubwa,2026-05-30 13:25:47


In [5]:
df = df_original.copy()
df.head()

,Bedrooms,Bedrooms(category),Property Category,Price,Price(with currency),District,Scraped_At
0,1,1 bedroom,"Studio Apartment in Arab Road, Kubwa for rent",600000,"₦ 600,000",Kubwa,2026-05-30 13:25:47
1,1,1 bedroom,"1bdrm Block of Flats in Arab Road, Kubwa for rent",1500000,"₦ 1,500,000",Kubwa,2026-05-30 13:25:47
2,1,1 bedroom,"Studio Apartment in Efab City Estate, Life Cam...",1200000,"₦ 1,200,000",Life Camp,2026-05-30 13:25:47
3,1,1 bedroom,Furnished 1bdrm Apartment in Las Colinas Villa...,1250000,"₦ 1,250,000",Bunkoro,2026-05-30 13:25:47
4,1,1 bedroom,"1bdrm Apartment in Arab Road, Kubwa for rent",1200000,"₦ 1,200,000",Kubwa,2026-05-30 13:25:47


In [6]:
columns_to_drop = ['Bedrooms(category)', 'Price(with currency)']
df.drop(columns=columns_to_drop, errors='ignore', inplace=True)

In [7]:
df.rename(columns={'Price': 'Price (Per Annum)'}, inplace=True)

In [8]:
df['Location'] = df['Property Category']

In [9]:
import numpy as np
def parse_property_type(text):
    if pd.isna(text):
        return np.nan
    
    t = text.lower()
    
    if 'terrace' in t or 'townhouse' in t:
        return 'Terraced Duplex'
    elif 'semi-detached' in t or 'semi detached' in t:
        return 'Semi-Detached Duplex'
    elif 'detached duplex' in t:
        return 'Detached Duplex'
    elif 'duplex' in t:
        return 'Detached Duplex'  
    elif 'detached bungalow' in t:
        return 'Detached Bungalow'
    elif 'semi-detached bungalow' in t or 'semi detached bungalow' in t:
        return 'Semi-detached Bungalow'
    elif 'terraced bungalow' in t or 'terrace bungalow' in t:
        return 'Terraced Bungalow'
    elif 'studio' in t or 'self contain' in t:
        return 'Self Contain'
    elif 'flat' in t or 'apartment' in t:
        return 'Apartment'
    
    return 'House'

df['Property Type'] = df['Location'].apply(parse_property_type)

broad_category_mapping = {
    'Apartment': 'Apartment',
    'Self Contain': 'Apartment',
    'Detached Duplex': 'Duplex',
    'Semi-Detached Duplex': 'Duplex',
    'Terraced Duplex': 'Duplex',
    'Detached Bungalow': 'Bungalow',
    'Semi-detached Bungalow': 'Bungalow',
    'Terraced Bungalow': 'Bungalow',
    'House': 'House'
}
df['Property Category'] = df['Property Type'].map(broad_category_mapping)

In [10]:
#standardize district column to amac districts only
amac_districts = [
    'Jabi', 'Kaura', 'Garki', 'Kabusa', 'City Centre', 'Wuse', 'Gwarinpa', 
    'Gui', 'Karshi', 'Asokoro', 'Jahi', 'Guzape', 'Apo', 'Durumi', 'Lugbe', 
    'Lokogoma', 'Maitama', 'Wuye', 'Katampe', 'Life Camp', 'Utako', 'Mabushi', 
    'Idu', 'Kado', 'Galadimawa', 'Karu', 'Gaduwa', 'Gudu', 'kukwaba', 'Karmo', 
    'Kubwa', 'Central Business District', 'CBD'
]

df['District'] = df['District'].str.strip()
amac_lower = [district.lower() for district in amac_districts]

df = df[df['District'].str.lower().isin(amac_lower)]

df['District'] = df['District'].apply(lambda x: 'CBD' if x.lower() in ['cbd', 'central business district'] else x.title())



In [11]:
new_order = [
    'District', 
    'Bedrooms', 
    'Price (Per Annum)', 
    'Property Category', 
    'Property Type', 
    'Location',   
    'Scraped_At'    
]

df = df[new_order]

df.head()

,District,Bedrooms,Price (Per Annum),Property Category,Property Type,Location,Scraped_At
0,Kubwa,1,600000,Apartment,Self Contain,"Studio Apartment in Arab Road, Kubwa for rent",2026-05-30 13:25:47
1,Kubwa,1,1500000,Apartment,Apartment,"1bdrm Block of Flats in Arab Road, Kubwa for rent",2026-05-30 13:25:47
2,Life Camp,1,1200000,Apartment,Self Contain,"Studio Apartment in Efab City Estate, Life Cam...",2026-05-30 13:25:47
4,Kubwa,1,1200000,Apartment,Apartment,"1bdrm Apartment in Arab Road, Kubwa for rent",2026-05-30 13:25:47
5,Life Camp,2,1140000,Apartment,Apartment,Furnished 2bdrm Apartment in Life Camp for rent,2026-05-30 13:25:47


In [15]:
num_isnull = df.isnull().sum()
print(num_isnull)

District             0
Bedrooms             0
Price (Per Annum)    0
Property Category    0
Property Type        0
Location             0
Scraped_At           0
dtype: int64


In [17]:
df.to_csv(BASE_DIR / "Data" / "processed" / "jiji_cleaned.csv", index=False)